# `projection_ssp`

> Generated from `config/ais_0.1.py` by `config/make_step_notebooks.py` on 2026-10-28. Edit the driver and regenerate; changes made here are overwritten.

HO projections from the historical end state. Each chunk resubmits its successor through `launch_projection_ho.pbs`, so later chunks run the driver, not this notebook.

**Reads** (`load_model` calls in this step):
- `resume_file`
- `hist_path = f'{model_dir}/AIS3_historical_1995_2019{hist_tag}.nc'`

**Writes** (`save_model` calls in this step):
- `init_path + '.tmp'`
- `ckpt + '.tmp'`

**Run as a batch job** (from `config/notebooks/`; resources copied from `config/launch_projection_ho.pbs`):

```bash
qsub -q hugemem -l ncpus=24,mem=720GB,walltime=45:00:00 -v NB=ais_0.1/17_projection_ssp.ipynb launch_notebook.pbs
```

Add run options to the `-v` list, e.g. `NB=...,AIS3_PROJ_SCENARIO=<value>`. The executed copy is written next to this notebook as `executed/<name>.<jobid>.ipynb`. Interactive use needs a Jupyter session with at least those resources.

**Run options**: set them in the next cell (interactive) or with `qsub -v` (batch).

Status, results and known failures: [`docs/run_guide.md`](../../../docs/run_guide.md) and the worklogs in `docs/`.

In [ ]:
import os

# Uncomment to set an option for an interactive run. Values passed with
# `qsub -v` already arrive in the environment. Defaults shown are the code's own.
# os.environ['AIS3_PROJ_SCENARIO'] = ...   # code default: 'ctrl'
# os.environ['AIS3_PROJ_GCM'] = ...   # code default: ''
# os.environ['AIS3_PROJ_FORCING'] = ...   # code default: 'both'
# os.environ['AIS3_PROJ_MELT'] = ...   # code default: 'ref'
# os.environ['AIS3_PROJ_HIST_TAG'] = ...   # code default: '_relaxfix'
# os.environ['AIS3_PROJ_END'] = ...   # code default: '2100'
# os.environ['AIS3_PROJ_SMOKE'] = ...   # code default: ''
# os.environ['AIS3_PROJ_COLLAPSE'] = ...   # code default: ''

## Setup

The preamble of `ais_0.1.py`: imports, helpers, settings and the data catalog.

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os
import glob
import copy
from pathlib import Path


def racmo_mean_smb(md, y0, y1):
    """Time-mean RACMO2.4p1 smbgl over calendar years y0..y1 on md's mesh, in m ice eq/yr.

    Same unit handling as historical_dhdt_tuning (monthly kg/m^2 summed, divided by rho_ice),
    averaged over the years. Uses the module-level `catalog`.
    """
    data = catalog.load_dataset('racmo2.4p1_monthly_11km_1979-2023')
    smbgl = data['smbgl']
    rx, ry = pyissm.tools.general.ll_to_xy(data['lat'].values, data['lon'].values, -1)
    sel = smbgl.sel(time=(smbgl['time.year'] >= y0) & (smbgl['time.year'] <= y1))
    n_years = y1 - y0 + 1
    assert sel.sizes['time'] == 12 * n_years, f"expected {12 * n_years} months, got {sel.sizes['time']}"
    mean_myr = (sel.sum('time') / n_years / md.materials.rho_ice).squeeze('height').to_numpy()
    out = pyissm.data.interp.points_to_mesh(rx, ry, mean_myr, md.mesh.x, md.mesh.y)
    n_nan = int(np.isnan(out).sum())
    print(f"   RACMO {y0}-{y1} mean SMB: mesh-mean {np.nanmean(out):.3f} m ice eq/yr, "
          f"{n_nan} vertices outside RACMO coverage set to 0")
    return np.nan_to_num(out, nan=0.0)


def sync_mesh_z(md):
    """Recompute the extruded mesh's vertex z from the current geometry (base + sigma*H).

    Keeps each column's relative layer spacing (sigma from the existing z), so it works even if
    md.mesh.z is stale -- e.g. after geometry was edited without touching z, as the thickness
    restoration (restore_floored_thickness.py) and the historical chunk resume did.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    n_layers = nv // nvb
    assert n_layers * nvb == nv
    assert np.asarray(md.mesh.vertexonbase).astype(bool).ravel()[:nvb].all()
    assert np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()[-nvb:].all()
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(n_layers, nvb)
    span = z[-1] - z[0]
    safe = np.abs(span) > 1e-9
    sigma = np.where(safe[None, :], (z - z[0]) / np.where(safe, span, 1.0)[None, :],
                     np.linspace(0.0, 1.0, n_layers)[:, None])
    base = np.asarray(md.geometry.base, dtype=float).ravel()[:nvb]
    thick = np.asarray(md.geometry.thickness, dtype=float).ravel()[:nvb]
    d_base = np.abs(z[0] - base)
    d_surf = np.abs(z[-1] - (base + thick))
    print(f"   mesh.z vs geometry before sync: base max {d_base.max():.3f} m ({int((d_base > 0.01).sum())} "
          f"columns off by >1 cm), surface max {d_surf.max():.3f} m ({int((d_surf > 0.01).sum())} columns)")
    md.mesh.z = (base[None, :] + sigma * thick[None, :]).ravel()


def adopt_final_state(md):
    """Make the transient's last step the model's own state before save_model.

    solve() fills md.results only, so without this save_model writes the transient's INPUT
    geometry -- the reason neither relaxation ever reached downstream steps (worklog 4y).
    Carries Thickness and MaskOceanLevelset; base/surface are recomputed from them with the
    hydrostatic-floating/bed-grounded formula (raw output Base can sit below bed at newly
    grounded vertices, worklog 4o); velocities go to md.initialization when saved.
    """
    ts = md.results.TransientSolution
    nv = md.mesh.numberofvertices

    def last(field):
        if isinstance(ts, list):
            return np.asarray(getattr(ts[-1], field), dtype=float).ravel().copy()
        raw = np.asarray(getattr(ts, field), dtype=float).ravel()
        return raw.reshape(raw.size // nv, nv)[-1].copy()

    thick = last('Thickness')
    ol = last('MaskOceanLevelset')
    bed = np.asarray(md.geometry.bed, dtype=float).ravel()
    ri = float(md.materials.rho_ice)
    rw = float(md.materials.rho_water)
    base = np.maximum(np.where(ol >= 0, bed, -thick * ri / rw), bed)
    old_thick = np.asarray(md.geometry.thickness, dtype=float).ravel()
    old_ol = np.asarray(md.mask.ocean_levelset, dtype=float).ravel()
    md.geometry.thickness = thick
    md.geometry.base = base
    md.geometry.surface = base + thick
    md.mask.ocean_levelset = ol
    set_vel = []
    for field in ('Vx', 'Vy', 'Vz', 'Vel'):
        try:
            setattr(md.initialization, field.lower(), last(field))
            set_vel.append(field)
        except (AttributeError, ValueError):
            pass
    d = thick - old_thick
    print(f"   adopted final transient state: mean|dH| {np.abs(d).mean():.3f} m, max|dH| "
          f"{np.abs(d).max():.2f} m; grounded/floating changed at "
          f"{100 * np.mean((ol >= 0) != (old_ol >= 0)):.3f}% of vertices; initialization set: {set_vel}")


def rescale_rheology_B_d2(md, B_ref_2d):
    """Rescale md's 3D rheology_B column by column to a 2D reference B, keeping the vertical
    profile ("D2", worklog 5c).

    Grounded columns match the deformation-weighted column B, B_eff = (sum w B^-n / sum w)^(-1/n)
    with w = (1 - sigma)^(n+1) (sigma 0 at the bed), so the soft basal layers count as they do for
    shear flow; floating columns match the plain column mean (plug flow). HO's steady-state
    temperature makes its basal ice much softer than the reference, which made HO flow ~9% too
    fast; with this rescale a diagnostic solve gave grounded surface speed / observed median 1.00.
    Idempotent: a column already matching the reference gets scale 1.
    """
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    nl = nv // nvb
    B_ref_2d = np.asarray(B_ref_2d, dtype=float).ravel()
    assert B_ref_2d.size == nvb, f'reference B has {B_ref_2d.size} values, base layer has {nvb}'
    z = np.asarray(md.mesh.z, dtype=float).ravel().reshape(nl, nvb)
    span = z[-1] - z[0]
    sig = (z - z[0]) / np.where(np.abs(span) > 1e-9, span, 1.0)[None, :]
    w = np.diff(sig, axis=0)
    wsum = w.sum(axis=0)
    Bh = np.asarray(md.materials.rheology_B, dtype=float).ravel().reshape(nl, nvb)
    n = float(np.unique(np.asarray(md.materials.rheology_n).ravel())[0])
    B_mean = (0.5 * (Bh[1:] + Bh[:-1]) * w).sum(axis=0) / np.maximum(wsum, 1e-12)
    wdef = (1.0 - sig) ** (n + 1.0)
    wd = 0.5 * (wdef[1:] + wdef[:-1]) * w
    A_mid = 0.5 * (Bh[1:] ** -n + Bh[:-1] ** -n)
    B_eff = ((wd * A_mid).sum(axis=0) / np.maximum(wd.sum(axis=0), 1e-30)) ** (-1.0 / n)
    grounded = np.asarray(md.mask.ocean_levelset).ravel()[:nvb] > 0
    scale = np.where(grounded, B_ref_2d / B_eff, B_ref_2d / B_mean)
    scale = np.where(wsum > 0, scale, 1.0)
    md.materials.rheology_B = (Bh * scale[None, :]).ravel()
    print(f"   D2 rheology rescale: grounded median {np.median(scale[grounded]):.3f} "
          f"(p10 {np.percentile(scale[grounded], 10):.3f}, p90 {np.percentile(scale[grounded], 90):.3f}), "
          f"floating median {np.median(scale[~grounded]):.3f}")


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'

# List all steps for clarity
all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
    'projection_ssp',
]

# Define steps to run
# steps = ['process_domain', 'mesh', 'param']
# steps = ['mesh', 'param']
# steps = ['ssa_rheology_floating_inv_sensit']
# steps = ['ssa_rheology_floating_inv_lcurve']
# steps = ['ssa_friction_forward_check']
# steps = ['ssa_friction_forward_check_budd']
# steps = ['param']
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
# steps = ['param']
# steps = ['param']
# steps = ['ssa_inverted_solve']
# steps = ['ssa_relaxation']
steps = ['ho_friction_inv']
# Per-job override via `qsub -v AIS3_STEPS=a,b` -- lets a launcher pick its step without editing
# the shared `steps` above and holding that edit through a multi-day hugemem queue wait.
if os.environ.get('AIS3_STEPS'):
    steps = os.environ['AIS3_STEPS'].split(',')
# Suffix for historical_dhdt_tuning's checkpoint file and execution name, so a rerun
# (e.g. AIS3_HIST_TAG=_chunk1rerun) cannot overwrite the finished AIS3_historical_1995_2019.nc.
HIST_TAG = os.environ.get('AIS3_HIST_TAG', '')
# steps = ['ssa_friction_inv_lcurve']
# steps = ['ssa_friction_inv_sensit']

#### Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

#### Initialise Data Catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Step `projection_ssp`

In [ ]:
steps = ['projection_ssp']  # this notebook's step; overrides the driver default and AIS3_STEPS

**From the driver:**

TODO: actual tuning loop. This step reports the mismatch for ONE (gamma_0, cf501) configuration; turning this into a real calibration means re-running stage 3 (melt_gamma_tuning) and/or ssa_friction_inv_reg_lcurve at a few different values and repeating this comparison, keeping whichever configuration minimises mismatch_rmse -- mirroring the L-curve/sensitivity pattern already used elsewhere in this pipeline (build_parameter_grid over the candidate values, rerun this dH/dt comparison per candidate) rather than a new methodology. Not done automatically here since each candidate requires re-running the full multi-stage chain (friction/melt/relaxation/historical), which is a substantial compute cost to automate blindly rather than direct. NOTE: AIS3_historical_1995_2019.nc was already saved above as this final chunk's checkpoint -- it now holds the completed run, no separate save needed here.

### Stage 6: Future projections (CTRL and SSP-forced), HO mesh

HO counterpart of ais_0.1_SSA.py's projection_ssp_ssa (same forcing, same experiment names with an ho_ prefix on files), for the few HO runs that check the SSA ensemble: ctrl, ssp585 ref and optionally ssp585 high melt, to 2100 (docs/projection_experiments.md). One experiment per job chain, chosen with qsub -v via launch_projection_ho.pbs:

AIS3_PROJ_SCENARIO / AIS3_PROJ_GCM / AIS3_PROJ_FORCING / AIS3_PROJ_MELT / AIS3_PROJ_SMOKE as in the SSA step; AIS3_PROJ_MELT=high is HO refit candidate g7 (gamma_0=15000, models/deltaT_refit_state.json) AIS3_PROJ_END       final year (default 2100) AIS3_PROJ_HIST_TAG  which HO historical run to start from (default _relaxfix, i.e. AIS3_historical_1995_2019_relaxfix.nc, which must reach 2019) AIS3_PROJ_COLLAPSE  not available on HO yet (SSA step only)

Forcing: TF0/SMB0 from the HO historical run, anomalies relative to 1995-2014 built on the base layer with the SSA step's interpolation and copied up every column (the HO base layer is the SSA mesh, checked below), so HO and SSA see identical forcing. GCM climatologies on that mesh are shared with the SSA step (models/AIS3_proj_ssa_*clim_*.npy).

Chunks of 10 yr (HO solves ~1.37 h per model year on 96 hugemem cpus, and 96-cpu hugemem jobs are capped at 24 h). Output is annual; each chunk writes a new checkpoint AIS3_proj_ho_{exp}_{year}.nc and a history npz (base-layer thickness/levelset/melt, base and surface speed, scalars), then qsubs the next chunk unless models/AIS3_proj_ho_{exp}.STOP exists.

In [ ]:
print("-------------------------------------------------------------")
print(" FUTURE PROJECTION, HO (ANOMALY-FORCED, CHUNKED)")

In [ ]:
print("-------------------------------------------------------------")
import subprocess
import re
import json
import netCDF4
from scipy.spatial import Delaunay

FORCING_ROOT = '/g/data/au88/ismip6/2300/forcings/ISMIP7/AIS'
SMB_PARAM = 'dEBM2-8000m'
CLIM_Y0, CLIM_Y1 = 1995, 2014
START_YEAR = 2019.0
TIME_STEP = 0.1
TIME_SENTINEL = 1e9
CHUNK_YEARS = 10.0
HO_MIN_PER_YEAR = 82.0  # 96-cpu hugemem solve, historical run (6 yr in 8.1-8.3 h)

scenario = os.environ.get('AIS3_PROJ_SCENARIO', 'ctrl')
gcm = os.environ.get('AIS3_PROJ_GCM', '')
forcing_mode = os.environ.get('AIS3_PROJ_FORCING', 'both')
melt_choice = os.environ.get('AIS3_PROJ_MELT', 'ref')
MELT_MEMBERS = {'high': 'g7'}
end_year = float(os.environ.get('AIS3_PROJ_END', '2100'))
hist_tag = os.environ.get('AIS3_PROJ_HIST_TAG', '_relaxfix')
smoke = os.environ.get('AIS3_PROJ_SMOKE', '') == '1'
assert forcing_mode in ('both', 'ocean', 'smb'), forcing_mode
if scenario == 'ctrl':
    exp = f'ctrl_{melt_choice}' if melt_choice != 'ref' else 'ctrl'
    use_ocean = use_smb = False
else:
    assert gcm, 'AIS3_PROJ_GCM is required for an SSP experiment'
    ftag = {'both': 'ref', 'ocean': 'oceanonly', 'smb': 'smbonly'}[forcing_mode]
    exp = f"{scenario}_{gcm.split('-')[0].lower()}_{melt_choice}_{ftag}"
    use_ocean = forcing_mode in ('both', 'ocean')
    use_smb = forcing_mode in ('both', 'smb')
if os.environ.get('AIS3_PROJ_COLLAPSE', '') == '1':
    # The SSA step's levelset version (sub-solves split at mask changes) is not ported to the
    # prism mesh yet; the extra-melt version tried first was unstable (worklog 6.12).
    raise NotImplementedError('AIS3_PROJ_COLLAPSE: SSA track only for now (ais_0.1_SSA.py)')

In [ ]:
print(f"-- Experiment ho_{exp}: scenario={scenario} gcm={gcm or '-'} forcing={forcing_mode} "
      f"melt={melt_choice} end={end_year:.0f} start file tag={hist_tag!r} smoke={smoke}")

init_path = f'{model_dir}/AIS3_proj_ho{hist_tag}_init.nc'
baseline_path = f'{model_dir}/AIS3_proj_ho{hist_tag}_baseline.npz'
stop_path = f'{model_dir}/AIS3_proj_ho{hist_tag}_{exp}.STOP'

def _interp_weights(src_x, src_y, tgt_x, tgt_y):
    """Same precomputed linear interpolation as ais_0.1_SSA.py (Delaunay + barycentric)."""
    src_xy = np.column_stack([np.asarray(src_x).ravel(), np.asarray(src_y).ravel()])
    tgt_xy = np.column_stack([np.asarray(tgt_x).ravel(), np.asarray(tgt_y).ravel()])
    tri = Delaunay(src_xy)
    simplex = tri.find_simplex(tgt_xy)
    vertices = np.take(tri.simplices, simplex, axis=0)
    temp = np.take(tri.transform, simplex, axis=0)
    bary = np.einsum('njk,nk->nj', temp[:, :2, :], tgt_xy - temp[:, 2])
    return vertices, np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True))), simplex

def _interp_apply(values, vtx, wts, simplex):
    out = np.einsum('nj,nj->n', np.asarray(values).ravel()[vtx], wts)
    out[simplex == -1] = np.nan
    return out

# -- Projection initial state, built once from the HO historical run's 2019 end state -------
if not os.path.exists(init_path):
    hist_path = f'{model_dir}/AIS3_historical_1995_2019{hist_tag}.nc'
    print(f"-- Building HO projection initial state from {hist_path}...")
    md = pyissm.model.io.load_model(hist_path)
    assert abs(float(md.timestepping.final_time) - START_YEAR) < 1e-6, \
        f'{hist_path} only reaches t={md.timestepping.final_time}; finish the historical run first'
    md.basalforcings.tf = [np.asarray(t, dtype=float) for t in md.basalforcings.tf]
    adopt_final_state(md)
    sync_mesh_z(md)
    md.results.TransientSolution = []
    nv = md.mesh.numberofvertices
    nvb = int(md.mesh.numberofvertices2d)
    mb = np.asarray(md.smb.mass_balance, dtype=float)
    mb_years = mb[nv, :]
    sel = (mb_years >= CLIM_Y0) & (mb_years <= CLIM_Y1)
    assert sel.sum() == CLIM_Y1 - CLIM_Y0 + 1, f'historical SMB years {mb_years}'
    smb0_base = np.nan_to_num(mb[:nvb, sel].mean(axis=1), nan=0.0)
    tf0_base = np.stack([np.asarray(t, dtype=float)[:nvb, 0] for t in md.basalforcings.tf])
    tf_depths0 = np.asarray(md.basalforcings.tf_depths, dtype=float).ravel()
    print(f"   SMB0 = RACMO {CLIM_Y0}-{CLIM_Y1} mean: base-layer mean {smb0_base.mean():.4f} m ice eq/yr; "
          f"TF0 {tf0_base.shape}; gamma_0={float(md.basalforcings.gamma_0)}")
    nl = nv // nvb
    md.basalforcings.tf = [np.append(np.tile(tf0_base[k], nl), TIME_SENTINEL).reshape(-1, 1)
                           for k in range(tf0_base.shape[0])]
    md.smb = pyissm.model.classes.smb.default(md.smb)
    md.smb.mass_balance = np.tile(smb0_base, nl)
    md.timestepping.start_time = START_YEAR
    md.timestepping.final_time = START_YEAR
    np.savez(baseline_path + '.tmp.npz', tf0_base=tf0_base, tf_depths=tf_depths0, smb0_base=smb0_base)
    os.replace(baseline_path + '.tmp.npz', baseline_path)
    pyissm.model.io.save_model(md, init_path + '.tmp')
    os.replace(init_path + '.tmp', init_path)
    print(f"   Saved {init_path} and {baseline_path}")
    del md

base = np.load(baseline_path)
tf0_base, tf_depths0, smb0_base = base['tf0_base'], base['tf_depths'], base['smb0_base']

def set_static_forcing(md):
    nl = md.mesh.numberofvertices // tf0_base.shape[1]
    md.basalforcings.tf = [np.append(np.tile(tf0_base[k], nl), TIME_SENTINEL).reshape(-1, 1)
                           for k in range(tf0_base.shape[0])]
    md.basalforcings.tf_depths = tf_depths0.copy()
    md.smb = pyissm.model.classes.smb.default(md.smb)
    md.smb.mass_balance = np.tile(smb0_base, nl)

# -- Resume from this experiment's latest checkpoint, else the initial state ----------------
ckpts = []
for f in glob.glob(f'{model_dir}/AIS3_proj_ho{hist_tag}_{exp}_*.nc'):
    tail = Path(f).stem[len(f'AIS3_proj_ho{hist_tag}_{exp}_'):]
    if tail.isdigit():
        ckpts.append((float(tail), f))
ckpts.sort()
current_time, resume_file = (ckpts[-1] if (ckpts and not smoke) else (START_YEAR, init_path))
if current_time >= end_year:
    raise SystemExit(f"ho_{exp} already complete (checkpoint at {current_time:.0f})")
edges = [START_YEAR] + [float(e) for e in np.arange(2030.0, end_year, CHUNK_YEARS)] + [end_year]
t1 = next(e for e in edges if e > current_time)  # 2019->2030->2040->...->end
if smoke:
    t1 = current_time + 0.2

In [ ]:
print(f"-- Chunk {current_time:.1f} -> {t1:.1f} from {resume_file}")
md = pyissm.model.io.load_model(resume_file)
md.results.TransientSolution = []
nv = md.mesh.numberofvertices
nvb = int(md.mesh.numberofvertices2d)
nl = nv // nvb
assert abs(float(md.timestepping.final_time) - current_time) < 1e-6, \
    f'checkpoint final_time {md.timestepping.final_time} != {current_time}'
xb = np.asarray(md.mesh.x, dtype=float).ravel()[:nvb]
yb = np.asarray(md.mesh.y, dtype=float).ravel()[:nvb]

if melt_choice != 'ref':
    with open(f'{model_dir}/deltaT_refit_state.json') as f:
        cand = json.load(f)['candidates'][MELT_MEMBERS.get(melt_choice, melt_choice).lstrip('g')]
    assert cand['converged'], f'HO refit candidate {melt_choice} did not converge'
    md.basalforcings.gamma_0 = float(cand['gamma_0'])
    md.basalforcings.delta_t = np.asarray(cand['history'][-1]['delta_t'], dtype=float)
print(f"   melt: gamma_0={float(md.basalforcings.gamma_0)}, deltaT range "
      f"[{np.min(md.basalforcings.delta_t):.3f}, {np.max(md.basalforcings.delta_t):.3f}]")

# -- Forcing (base layer, then copied up each column) -------------------------------------
years = np.arange(int(np.floor(current_time)), int(np.ceil(t1)) + 1)
col_times = years + 0.5

def _tf_years(f):
    y = Path(f).stem.split('_')[-1].split('-')
    return int(y[0]), int(y[-1])

def _load_tf(gcm_, scen, y0, y1):
    files = [f for f in sorted(glob.glob(f'{FORCING_ROOT}/{gcm_}/{scen}/ocean/tf/v3/tf_AIS_*.nc'))
             if _tf_years(f)[1] >= y0 and _tf_years(f)[0] <= y1]
    if not files:
        raise FileNotFoundError(f'no TF files for {gcm_}/{scen} {y0}-{y1}')
    # cftime throughout: proleptic-Gregorian files (MRI-ESM2-0) otherwise decode to pandas timestamps
    # before ~2262 and cftime after, which open_mfdataset cannot combine.
    da = xr.open_mfdataset(files, combine='by_coords',
                           decode_times=xr.coders.CFDatetimeCoder(use_cftime=True))['tf']
    return da.sel(time=(da['time.year'] >= y0) & (da['time.year'] <= y1))

def _smb_year_myr(gcm_, scen, yr, rho_ice, yts):
    m = glob.glob(f'{FORCING_ROOT}/{gcm_}/{scen}/{SMB_PARAM}/acabf/v*/acabf_AIS_{gcm_}_{scen}_{SMB_PARAM}_v*_{yr}.nc')
    if not m:
        return None
    return (xr.open_dataset(m[0])['acabf'].mean('time') * yts / rho_ice).to_numpy()

if use_ocean or use_smb:
    # The HO base layer must be the SSA mesh for the shared climatology caches (worklog 4w).
    ssa_nc = netCDF4.Dataset(f'{model_dir}/AIS3_proj_ssa_init.nc')
    assert np.allclose(np.asarray(ssa_nc['mesh']['x'][:]).ravel(), xb) and \
        np.allclose(np.asarray(ssa_nc['mesh']['y'][:]).ravel(), yb), 'HO base layer != SSA mesh'
    ssa_nc.close()
    ref_grid = xr.open_dataset(sorted(glob.glob(f'{FORCING_ROOT}/{gcm}/historical/ocean/tf/v3/*.nc'))[0])
    gxx, gyy = np.meshgrid(ref_grid['x'].values, ref_grid['y'].values)
    vtx, wts, simplex = _interp_weights(gxx, gyy, xb, yb)
    to_mesh = lambda a: _interp_apply(a, vtx, wts, simplex)

if use_ocean:
    clim_path = f'{model_dir}/AIS3_proj_ssa_tfclim_{gcm}_{CLIM_Y0}-{CLIM_Y1}.npy'
    if not os.path.exists(clim_path):
        clim = _load_tf(gcm, 'historical', CLIM_Y0, CLIM_Y1)
        assert clim.sizes['time'] == CLIM_Y1 - CLIM_Y0 + 1, clim.sizes
        clim = clim.mean('time', skipna=True).to_numpy()
        np.save(clim_path + f'.{os.uname().nodename}.{os.getpid()}.tmp.npy', np.stack([to_mesh(clim[k]) for k in range(clim.shape[0])]))
        os.replace(clim_path + f'.{os.uname().nodename}.{os.getpid()}.tmp.npy', clim_path)
    tf_clim = np.load(clim_path)
    assert tf_clim.shape == tf0_base.shape, (tf_clim.shape, tf0_base.shape)
    tf_da = _load_tf(gcm, scenario, int(years[0]), int(years[-1]))
    tf_years = tf_da['time.year'].values
    assert np.allclose(tf_da['z'].values, tf_depths0), 'TF depth levels differ from TF0'
    tf_base = np.empty((tf0_base.shape[0], nvb, years.size))
    for j, yr in enumerate(years):
        src = yr if yr in tf_years else tf_years[tf_years <= yr].max()
        slab = tf_da.isel(time=int(np.where(tf_years == src)[0][0])).to_numpy()
        for k in range(tf0_base.shape[0]):
            tf_base[k, :, j] = tf0_base[k] + np.nan_to_num(to_mesh(slab[k]) - tf_clim[k], nan=0.0)
    fl = np.asarray(md.mask.ocean_levelset).ravel()[:nvb] < 0
    d = tf_base[:, fl, -1] - tf0_base[:, fl]
    print(f"      TF anomaly {years[-1]} over floating base vertices: mean {d.mean():+.3f} C, "
          f"p95 {np.percentile(d, 95):+.3f}")
    md.basalforcings.tf = []
    for k in range(tf0_base.shape[0]):
        col = np.empty((nv + 1, years.size))
        col[:nv] = np.tile(tf_base[k], (nl, 1))
        col[nv] = col_times
        md.basalforcings.tf.append(col)
    md.basalforcings.tf_depths = tf_depths0.copy()
    del tf_base
else:
    md.basalforcings.tf = [np.append(np.tile(tf0_base[k], nl), TIME_SENTINEL).reshape(-1, 1)
                           for k in range(tf0_base.shape[0])]
    md.basalforcings.tf_depths = tf_depths0.copy()

md.smb = pyissm.model.classes.smb.default(md.smb)
if use_smb:
    rho_ice, yts = float(md.materials.rho_ice), float(md.constants.yts)
    sclim_path = f'{model_dir}/AIS3_proj_ssa_smbclim_{gcm}_{SMB_PARAM}_{CLIM_Y0}-{CLIM_Y1}.npy'
    if not os.path.exists(sclim_path):
        acc = [_smb_year_myr(gcm, 'historical', y, rho_ice, yts) for y in range(CLIM_Y0, CLIM_Y1 + 1)]
        assert all(a is not None for a in acc), 'missing historical acabf years'
        np.save(sclim_path + f'.{os.uname().nodename}.{os.getpid()}.tmp.npy', to_mesh(np.mean(acc, axis=0)))
        os.replace(sclim_path + f'.{os.uname().nodename}.{os.getpid()}.tmp.npy', sclim_path)
    smb_clim = np.load(sclim_path)
    mb = np.empty((nv + 1, years.size))
    last_a = None
    for j, yr in enumerate(years):
        a = _smb_year_myr(gcm, scenario, yr, rho_ice, yts)
        if a is None:
            assert last_a is not None, f'no acabf for {gcm}/{scenario} {yr}'
            a = last_a
        last_a = a
        mb[:nv, j] = np.tile(smb0_base + np.nan_to_num(to_mesh(a) - smb_clim, nan=0.0), nl)
        mb[nv, j] = col_times[j]
    print(f"      SMB anomaly {years[-1]}: base-layer mean {np.mean(mb[:nvb, -1] - smb0_base):+.4f} m ice eq/yr")
    md.smb.mass_balance = mb
else:
    md.smb.mass_balance = np.tile(smb0_base, nl)

# -- Solver setup, same physics as historical_dhdt_tuning -----------------------------------
md.inversion.iscontrol = 0
md.verbose.solution = 1
md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
md.transient.isstressbalance = 1
md.transient.ismasstransport = 1
md.transient.issmb = 1
md.transient.isthermal = 0
md.transient.isgroundingline = 1
md.groundingline.migration = 'SubelementMigration'
md.transient.requested_outputs = [
    'Thickness', 'Surface', 'Base', 'MaskOceanLevelset', 'Vx', 'Vy', 'Vz', 'Vel',
    'BasalforcingsFloatingiceMeltingRate', 'SmbMassBalance',
    'IceVolume', 'IceVolumeAboveFloatation', 'GroundedArea', 'FloatingArea',
    'TotalSmb', 'TotalFloatingBmb', 'TotalGroundedBmb', 'GroundinglineMassFlux',
]  # no IcefrontMassFlux: ISSM has it for 2D elements only ("not implemented" on prisms, HO smoke test)
md.timestepping.start_time = current_time
md.timestepping.final_time = t1
md.timestepping.time_step = TIME_STEP
md.settings.output_frequency = 1 if smoke else int(round(1.0 / TIME_STEP))
md.settings.solver_residue_threshold = 1e-3
sync_mesh_z(md)

run_name = f'AIS3_proj_ho{hist_tag}_{exp}_{current_time:.0f}-{t1:.0f}' + ('_smoke' if smoke else '')
md.miscellaneous.name = run_name
md.cluster = pyissm.model.classes.cluster.gadi()
for attr in ('codepath', 'executionpath', 'storage', 'moduleuse', 'moduleload', 'login', 'project'):
    setattr(md.cluster, attr, getattr(cluster, attr))
md.cluster.queue = 'hugemem'
md.cluster.np = 96
md.cluster.memory = 1450 * 2
# 1.7x the historical run's rate, capped under the 24 h limit for 96-cpu hugemem jobs.
md.cluster.time = 60 if smoke else int(min(23.5 * 60, np.ceil(1.7 * (t1 - current_time) * HO_MIN_PER_YEAR)))
wall_min = 45 * 60
try:
    qs = subprocess.run(['qstat', '-f', os.environ['PBS_JOBID']], capture_output=True, text=True).stdout
    hh, mm, _ = re.search(r'Resource_List.walltime = (\d+):(\d+):(\d+)', qs).groups()
    wall_min = int(hh) * 60 + int(mm)
except (KeyError, AttributeError):
    pass
md.settings.waitonlock = max(30, wall_min - 120)
print(f"   inner solve time {md.cluster.time} min; wait budget {md.settings.waitonlock} min")

In [ ]:
print(f"-- Submitting and waiting on {run_name}...")
md = pyissm.model.execute.solve(md, 'Transient', load_only=False, runtime_name=False)

# -- History (annual): base layer, plus surface speed -----------------------------------
ts = md.results.TransientSolution
steps_list = list(ts.steps) if hasattr(ts, 'steps') else (ts if isinstance(ts, list) else [ts])
times = np.array([float(s.time) for s in steps_list])
# The solve must have reached t1. A solve killed at its walltime leaves a partial outbin that
# pyissm loads without complaint; adopting its last step and saving it as t1 silently skipped
# 19, 13 and 4 model years in three runs (worklog 6.10). Stop here instead: no history, no
# checkpoint, no next chunk.
if times.size == 0 or abs(times[-1] - t1) > 1e-6:
    raise SystemExit(f"{run_name}: solve ended at t={times[-1] if times.size else 'none'}, not {t1} "
                     f"(inner job killed or failed?) -- no checkpoint written, chain stopped")

def _stack(field, layer='base'):
    sl = slice(0, nvb) if layer == 'base' else slice(nv - nvb, nv)
    return np.stack([np.asarray(getattr(s, field), dtype=np.float32).ravel()[sl] for s in steps_list])

scalar_names = ['IceVolume', 'IceVolumeAboveFloatation', 'GroundedArea', 'FloatingArea', 'TotalSmb',
                'TotalFloatingBmb', 'TotalGroundedBmb', 'GroundinglineMassFlux']
scalars = {n: np.array([float(np.asarray(getattr(s, n, np.nan)).ravel()[0]) for s in steps_list])
           for n in scalar_names}
hist_out = f'{model_dir}/{run_name}_history.npz'
np.savez(hist_out, times=times, thickness=_stack('Thickness'), ocean_levelset=_stack('MaskOceanLevelset'),
         vel=_stack('Vel', 'surface'), vel_base=_stack('Vel'), melt=_stack('BasalforcingsFloatingiceMeltingRate'),
         smb=_stack('SmbMassBalance', 'surface'), **{f'scalar_{n}': v for n, v in scalars.items()})
vaf = scalars['IceVolumeAboveFloatation']
print(f"   saved {hist_out} ({times.size} outputs, t={times[0]:.2f}..{times[-1]:.2f})")
print(f"   VAF change over chunk {vaf[-1] - vaf[0]:+.4e} m^3 "
      f"(= {-(vaf[-1] - vaf[0]) * float(md.materials.rho_ice) / 1028.0 / 3.625e14 * 1000:+.2f} mm SLE)")
for n in ('GroundedArea', 'TotalSmb', 'TotalFloatingBmb', 'GroundinglineMassFlux'):
    print(f"   {n}: first {scalars[n][0]:.4e}, last {scalars[n][-1]:.4e}")

if smoke:
    print("-- Smoke test complete (no checkpoint, no chaining).")
else:
    adopt_final_state(md)
    sync_mesh_z(md)
    md.results.TransientSolution = []
    set_static_forcing(md)
    ckpt = f'{model_dir}/AIS3_proj_ho{hist_tag}_{exp}_{t1:.0f}.nc'
    pyissm.model.io.save_model(md, ckpt + '.tmp')
    os.replace(ckpt + '.tmp', ckpt)
    print(f"   saved checkpoint {ckpt}")
    if t1 < end_year and not os.path.exists(stop_path):
        env = ','.join(f'{k}={os.environ[k]}' for k in
                       ('AIS3_STEPS', 'AIS3_PROJ_SCENARIO', 'AIS3_PROJ_GCM', 'AIS3_PROJ_FORCING',
                        'AIS3_PROJ_MELT', 'AIS3_PROJ_END', 'AIS3_PROJ_HIST_TAG') if os.environ.get(k))
        log = f'/g/data/au88/jh7060/ACCESS-AIS3/config/logs/proj_ho_{exp}_{t1:.0f}-next'
        out = subprocess.run(['qsub', '-v', env, '-N', f'ho_{exp}'[:15], '-o', log + '.out', '-e', log + '.err',
                              '/g/data/au88/jh7060/ACCESS-AIS3/config/launch_projection_ho.pbs'],
                             capture_output=True, text=True)
        print(f"   next chunk submitted: {out.stdout.strip()} {out.stderr.strip()}")
    elif t1 < end_year:
        print(f"   {stop_path} exists -- not submitting the next chunk")
    else:
        print(f"-- ho_{exp} complete through {end_year:.0f}")